In [1]:
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
download_service = Service()
driver = webdriver.Chrome(service=download_service)

driver.get("https://www.sklepopon.com/szukaj-opony?sezon=zimowe&rozmiar=205/55R16&ofs=0")
# driver.get("https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16")
# https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16#&&/wEXDAULcGNrX1ByTFNST0YFA+KUpAUVcGNrX0xzdFNTb3J0UGFyYW1ldGVyBQIzNwUOcGNrX1RyTFNTc25JZHMFATIFCXBja19UckxTUgUD4pSkBQdwY2tfQ1BnBQEyBQpwY2tfVHJMU1BUBQQxMTAwBQhwY2tfSU9GUAUCNjgFDHBja19UckxTU0lkcwUDMjUyBQtwY2tfVHJJT25seQUD4pSkBQpwY2tfVHJMU1BGBQMxOTEFDXBja19UckxTVFRJZHMFATIFFXBja19UckxTU29ydERpcmVjdGlvbgUBMtD9aDKZ+9GYcVUKukEj6XnJhNql
#https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16#&&/wEXDAULcGNrX1ByTFNST0YFA+KUpAUVcGNrX0xzdFNTb3J0UGFyYW1ldGVyBQIzNwUOcGNrX1RyTFNTc25JZHMFATIFCXBja19UckxTUgUD4pSkBQdwY2tfQ1BnBQEzBQpwY2tfVHJMU1BUBQQxMTAwBQhwY2tfSU9GUAUCNjgFDHBja19UckxTU0lkcwUDMjUyBQtwY2tfVHJJT25seQUD4pSkBQpwY2tfVHJMU1BGBQMxOTEFDXBja19UckxTVFRJZHMFATIFFXBja19UckxTU29ydERpcmVjdGlvbgUBMlWnHdgz3AaHVlQ3gBlfXfTWethI


In [3]:

import time

btn_cookie = driver.find_element(By.CSS_SELECTOR, "#klaro > div > div > div > div > div > button")
btn_cookie.click()

time.sleep(5)
try:
    driver.execute_script("""
        const shadowHost = document.querySelector("body > div.gr-visual-prompt");
        const shadowRoot = shadowHost.shadowRoot;
        const closeButton = shadowRoot.querySelector("div > div:nth-child(2) > button:nth-child(1)");
        closeButton.click();
    """)
    print("Okienko powiadomień zostało zamknięte.")
except Exception as e:
    print("Nie udało się zamknąć okienka powiadomień:", e)

Okienko powiadomień zostało zamknięte.


In [15]:
import re
from selenium.common.exceptions import NoSuchElementException

opony_data_list = []

try:
    # Znalezienie wszystkich elementów opon w sekcji listing-products-element
    opony_elements = driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')
    
    # Iteracja przez każdy element opony
    for opona_element in opony_elements:
        # Słownik do przechowywania danych jednej opony
        opona_data = {}

        # Pobranie danych do słownika
        opona_data['name'] = opona_element.get_attribute('data-ee-product-properties').split(";")[0].split(":")[1]
        opona_data['price'] = float(opona_element.get_attribute('data-ee-product-properties').split(";")[3].split(":")[1])
        opona_data['brand'] = opona_element.get_attribute('data-ee-product-properties').split(";")[4].split(":")[1]
        opona_data['size'] = opona_element.get_attribute('data-ee-product-properties').split(";")[5].split(":")[1]
        opona_data['model'] = opona_element.get_attribute('data-ee-product-properties').split(";")[6].split(":")[1]
        
        # Pobieranie szczegółowych informacji o etykiecie
        etykieta_elements = opona_element.find_elements(By.CSS_SELECTOR, 'span.icon-fuel-new ~ span, span.icon-rain-new ~ span, span.icon-speaker-new ~ span')
        opona_data['fuel_index'] = etykieta_elements[0].text if len(etykieta_elements) > 0 else None
        opona_data['wet_grip_index'] = etykieta_elements[1].text if len(etykieta_elements) > 1 else None
        opona_data['noise_index'] = etykieta_elements[2].text.split(" ")[0] if len(etykieta_elements) > 2 else None
        
        # Pobieranie poziomu hałasu, uwzględniając wewnętrzny <span> z dB
        try:
            noise_level_elements = opona_element.find_elements(By.CSS_SELECTOR, "span.self-center.tracking-tighter.sm\\:tracking-normal")
            for noise_level_element in noise_level_elements:
                text = noise_level_element.text
                match = re.search(r'\d+', text)
                if match:
                    noise_level = int(match.group())
                else:
                    noise_level = None
                opona_data['noise_level'] = noise_level
        except (NoSuchElementException, IndexError):
            opona_data['noise_level'] = None
        
        class_mapping = {
            "Premium": "Premium",
            "Średnia": "Średnia",
            "Średniej": "Średnia",
            "Ekonomiczna": "Ekonomiczna",
            "Ekonomicznej": "Ekonomiczna"
        }
        try:
            klasa_element = opona_element.find_element(By.XPATH, ".//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'klas')]")
            # Dopasuj i wyczyść tekst
            opona_class = klasa_element.text.lower().replace("w klasie ", "").replace("klasa ", "").strip().capitalize()
            opona_data['class'] = class_mapping.get(opona_class, opona_class)
        except NoSuchElementException:
            opona_data['class'] = None
        
        # Pobieranie oceny użytkownika (tekstowa wartość obok gwiazdek)
        try:
            user_rating_element = opona_element.find_element(By.XPATH, ".//li[contains(@class, 'xl:hidden')]//span[contains(@class, 'ml-1')]")
            opona_data['user_rating'] = float(user_rating_element.text.replace(",", "."))
        except NoSuchElementException:
            opona_data['user_rating'] = None
        
        # Dodajemy dane opony do listy
        opony_data_list.append(opona_data)
        
    # Wydrukowanie listy wszystkich danych o oponach
    df = pd.DataFrame(opony_data_list)
    display(df)

finally:
    pass
    # Zamknięcie WebDrivera
    # driver.quit()


Premium
Średniej
Ekonomicznej
Ekonomiczna
Ekonomiczna
Premium
Premium
Ekonomiczna
Premium
Ekonomiczna
Premium
Ekonomiczna
Ekonomiczna
Premium
Średnia
Średnia
Średnia
Średnia
Premium
Premium
Średnia
Premium
Średnia


,name,price,brand,size,model,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating
0,Wintrac 205/55 R16 91 H,376.99,Vredestein,205/55 R16,Wintrac,C,B,B,70,Premium,5.4
1,Wintercraft WP52 205/55 R16 91 H,320.00,Kumho,205/55 R16,Wintercraft WP52,C,B,B,72,Średnia,5.1
2,DIMAX ALPINE 205/55 R16 94 H,225.49,Radar,205/55 R16,DIMAX ALPINE,D,C,A,69,Ekonomiczna,5.2
3,Frigo HP2 205/55 R16 91 H,283.00,Dębica,205/55 R16,Frigo HP2,C,C,B,72,Ekonomiczna,5.2
4,Frigo 2 205/55 R16 91 T,239.00,Dębica,205/55 R16,Frigo 2,C,C,B,71,Ekonomiczna,5.1
5,Winter i*cept RS3 W462 205/55 R16 91 T,322.00,Hankook,205/55 R16,Winter i*cept RS3 W462,C,B,B,72,Premium,5.3
6,Snowproof 1 205/55 R16 91 H,335.00,Nokian Tyres,205/55 R16,Snowproof 1,C,B,B,70,Premium,5.4
7,SW608 205/55 R16 91 H,244.99,Goodride,205/55 R16,SW608,C,C,B,72,Ekonomiczna,5.1
8,Snowproof 2 205/55 R16 91 H,385.00,Nokian Tyres,205/55 R16,Snowproof 2,C,B,A,69,Premium,5.6
9,Z507 205/55 R16 91 V,227.00,Goodride,205/55 R16,Z507,C,C,B,72,Ekonomiczna,5.1
